<a href="https://colab.research.google.com/github/chandupradeep18/Building-with-the-Claude-API/blob/main/Claude_ai_Bot_stream_response.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install anthropic

In [22]:
from anthropic import Anthropic
from datetime import datetime,date
from google.colab import userdata
import os
import csv
import json
import uuid

In [17]:
client=Anthropic(api_key=userdata.get('Claude'))
model='claude-haiku-4-5'
max_tokens=200

In [24]:
chat_history=list()
def user_message(message):
  user={"role":"user","content":message}
  details={"Date & time":datetime.now(),"role":"user","content":message}
  chat_history.append(user)
  Collect_chat_history(details)
def assistant_message(message):
  assistant={"role":"assistant","content":message}
  details={"Date & time":datetime.now(),"role":"assistant","content":message}
  chat_history.append(assistant)
  Collect_chat_history(details)
def Collect_response_details(details):
  file_exist=os.path.exists('/content/drive/MyDrive/ClaudeResponse/response_details.csv')
  with open('/content/drive/MyDrive/ClaudeResponse/response_details.csv','a',newline='') as save_response:
    writer=csv.DictWriter(save_response,fieldnames=details.keys())
    if not file_exist:
      writer.writeheader()
    writer.writerow(details)
def Collect_chat_history(details):
  file_exist=os.path.exists('/content/drive/MyDrive/ClaudeResponse/chat_history.csv')
  with open('/content/drive/MyDrive/ClaudeResponse/chat_history.csv','a',newline='') as save_response:
    writer=csv.DictWriter(save_response,fieldnames=details.keys())
    if not file_exist:
      writer.writeheader()
    writer.writerow(details)
def Save_chat_history(name,details):
  file_exist=os.path.exists('/content/drive/MyDrive/ClaudeResponse/'+name+'.json')
  with open('/content/drive/MyDrive/ClaudeResponse/'+name+'.json','w') as save_json:
    json.dump(details,save_json)

In [ ]:
chat_history=list()
json_data=[]
bot_role=input("Enter a Bot Role : ")
while True:
  user_input=input("Ask Ai any Thing : ")
  if not user_input:
    break
  system=None
  if bot_role:
    system=bot_role
  user_message(user_input)
  param={"model":model,"max_tokens":max_tokens,"messages":chat_history,"temperature":0.5,'system':system}
  with client.messages.stream(**param) as stream:
    for part in stream.text_stream:
      print(part,end="")
    response=stream.get_final_message()
    details={'Time':datetime.now().isoformat(),'id':response.id,'role':response.role,'stop_reason':response.stop_reason,'input_tokens':response.usage.input_tokens,'output_tokens':response.usage.output_tokens,'user':user_input,'assistant':response.content[0].text}
    assistant_message(response.content[0].text)
    Collect_response_details(details)
    json_data.append(details)
if json_data:
  Save_chat_history(str(uuid.uuid4()),json_data)
else:
  print('No Chat History')